In [1]:
import joblib
import numpy as np
import pandas as pd
import nflreadpy as nfl
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# true = all seasons; if faster run needed, use something like range(2006, 2027)
pbp_pandas = nfl.load_pbp(seasons=True).to_pandas()
print(pbp_pandas.shape)

(1296179, 372)


In [5]:
play_types = ["run", "pass", "punt", "field_goal"]

df = pbp_pandas[
    pbp_pandas["play_type"].isin(play_types)
    & pbp_pandas["posteam"].notna() #posteam means "team possessing the ball"
    & pbp_pandas["result"].notna()
    & (pbp_pandas["result"] != 0)
].copy()
#possessing team is the home team; boolean
df["posteam_is_home"] = (df["posteam"] == df["home_team"]).astype(int) 
#result column is margin of victory from perspective of home team. change the perspective to away team by flipping the sign if the possessing team is not the home team
pos_result = np.where(df["posteam_is_home"] == 1, df["result"], -df["result"]) 
#get whether the possessing team wins depending on whether the result, i.e. final score margin, is positive
df["posteam_win"] = (pos_result > 0).astype(int)
#converts play type column to fixed categories so that gradient boosting function can use it
df["play_type"] = pd.Categorical(df["play_type"].astype(str), categories=play_types)

features = [
    "game_seconds_remaining",
    "half_seconds_remaining",
    "down",
    "ydstogo",
    "yardline_100",
    "score_differential",
    "posteam_timeouts_remaining",
    "defteam_timeouts_remaining",
    "posteam_is_home",
]
features_play = features + ["play_type"]

required = ["posteam_win", "down", "ydstogo", "yardline_100", "score_differential", "game_seconds_remaining"]
model_df = df.dropna(subset=required)
train = model_df[model_df["season"] <= 2024]
test = model_df[model_df["season"] >= 2025]
y_train, y_test = train["posteam_win"], test["posteam_win"]
print(len(train), "train rows /", len(test), "test rows")
print(train["play_type"].value_counts())

960615 train rows / 50047 test rows
play_type
pass          501180
run           368978
punt           63923
field_goal     26534
Name: count, dtype: int64


comparing brier score loss on logistic vs gradient boosting vs adding play type to the latter

In [ ]:
"""LR_NUM = [c for c in STATE if c != "down"]


def make_logistic():
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), LR_NUM),
        ("down_ohe", OneHotEncoder(handle_unknown="ignore"), ["down"]),
    ])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])
"""
def make_gb():
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=100,
        l2_regularization=1.0, early_stopping=False,
        categorical_features="from_dtype",   # picks up the `play_type` category column
        random_state=42,
    )

"""lr_state = make_logistic().fit(train[features], y_train)
gb_state = make_gb().fit(train[features], y_train)"""
gb_play = make_gb().fit(train[features_play], y_train)

In [5]:
"""def metrics(name, y, p):
    return {"model": name, "brier": brier_score_loss(y, p), "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "auc": roc_auc_score(y, p)}

p_lr = lr_state.predict_proba(test[STATE])[:, 1]
p_gb_state = gb_state.predict_proba(test[STATE])[:, 1]
p_gb_play = gb_play.predict_proba(test[FEATURES_PLAY])[:, 1]

pd.DataFrame([
    metrics("logistic, state only", y_test, p_lr),
    metrics("boosting, state only", y_test, p_gb_state),
    metrics("boosting, state + play type", y_test, p_gb_play),
]).round(4)"""

,model,brier,log_loss,auc
0,"logistic, state only",0.1765,0.5209,0.8123
1,"boosting, state only",0.1738,0.5111,0.8170
2,"boosting, state + play type",0.1737,0.5109,0.8172


interpretation: adding play type to the gradient boosting algorithm does not significantly skew the results. gradient boosting is a bit better than logistic which is good because one of its most important benefits is it can run on cpu even for a large dataset, which is what we need within the scope of this projext

## win probaibility depending on next play
keep game state fixed and swap the play for each iteration; return all results. plays are limited to sensible options. run/pass on downs 1-3; punt/field goal only available on 4th down.

In [ ]:
def predict_wp_by_play(model, game_state):
    #game_state: dict with every key in STATE. return possession-team win probability for each allowed play.
    allowed = play_types if game_state["down"] == 4 else ["run", "pass"]
    rows = pd.DataFrame([game_state] * len(allowed))
    rows["play_type"] = pd.Categorical(allowed, categories=play_types)
    probs = model.predict_proba(rows[FEATURES_PLAY])[:, 1]
    return pd.Series(probs, index=allowed, name="posteam_wp")

#test based on a few different game states
second_and_six = {
    "game_seconds_remaining": 600, "half_seconds_remaining": 600, "down": 2, "ydstogo": 6,
    "yardline_100": 45, "score_differential": 2,
    "posteam_timeouts_remaining": 3, "defteam_timeouts_remaining": 3, "posteam_is_home": 1,
}
fourth_and_two = {
    "game_seconds_remaining": 600, "half_seconds_remaining": 600, "down": 4, "ydstogo": 2,
    "yardline_100": 35, "score_differential": -3,
    "posteam_timeouts_remaining": 2, "defteam_timeouts_remaining": 3, "posteam_is_home": 1,
}
print(predict_wp_by_play(gb_play, second_and_six).round(4), "\n")
print(predict_wp_by_play(gb_play, fourth_and_two).round(4))

In [ ]:
joblib.dump(gb_play, "wp_play_model.joblib")